# Diabetic Retinopathy — Training Control Panel

- Dataset: `data/btgraham-300/`; original official split membership is preserved.
- Current fast strategy: `undersampled`. Training keeps at most **2,500 examples per class**; validation and test are also reduced to approximate a **70:10:20** train/validation/test distribution.
- Run `prepare_undersampled_split.ipynb` once before training. It writes one manifest used by EfficientNet, weighted EfficientNet, DenseNet, fusion, and the evaluator.
- Expected sizes from the project's known counts: **9,024 train / 1,289 validation / 2,578 test**. Trust the actual counts printed by the preparation notebook if they differ.
- Undersampled runs use separate names such as `models/efficientnet_undersampled/`; older full-data runs/checkpoints are not overwritten.
- Metrics represent the fixed sampled validation/test subsets, not the complete official validation/test splits. Start training only by manually executing the corresponding training cell.


## 1. Environment and device check

In [1]:
from configs import tensorflow_setup as _tensorflow_setup

import sys
import numpy as np
import tensorflow as tf

_tensorflow_setup.configure_gpu_memory_growth(tf)

import tensorflow_datasets as tfds

print("Python:", sys.version.split()[0])
print("CUDA DLL directory:", _tensorflow_setup.CUDA_DLL_DIRECTORY)
print("TensorFlow:", tf.__version__)
print("Keras:", tf.keras.__version__ if hasattr(tf.keras, "__version__") else "not exposed")
print("NumPy:", np.__version__)
print("TFDS:", tfds.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))
print("EfficientNetB0 API:", hasattr(tf.keras.applications, "EfficientNetB0"))
print("Seed utility API:", hasattr(tf.keras.utils, "set_random_seed"))
from configs.config import TRAINING_SAMPLING_MODE, UNDERSAMPLE_MAX_PER_CLASS, SEED, training_model_name
print("Training sampling mode:", TRAINING_SAMPLING_MODE)
if TRAINING_SAMPLING_MODE == "undersampled":
    print("Maximum training examples per class:", UNDERSAMPLE_MAX_PER_CLASS)


Python: 3.10.20
CUDA DLL directory: C:\Users\kawal\miniconda3\envs\dr_tf\Library\bin
TensorFlow: 2.10.1
Keras: 2.10.0
NumPy: 1.23.5
TFDS: 4.8.3
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
EfficientNetB0 API: True
Seed utility API: True
Training sampling mode: undersampled
Maximum training examples per class: 2500


## 2. Data-loader smoke test

In [2]:
from scripts.data_loader import inspect_dataset, make_datasets, get_training_sampling_summary

inspect_dataset()
sampling_summary = get_training_sampling_summary(
    seed=SEED,
    training_sampling_mode=TRAINING_SAMPLING_MODE,
)
print("Original training class counts:", sampling_summary["original_training_class_counts"])
print("Retained training class counts:", sampling_summary["training_class_counts"])
print("Training examples retained:", sampling_summary["retained_training_examples"])
print("Validation examples retained:", sampling_summary.get("validation_examples_after_sampling"))
print("Test examples retained:", sampling_summary.get("test_examples_after_sampling"))
print("Training subset SHA-256:", sampling_summary.get("training_subset_sha256"))
print("Whole split-manifest SHA-256:", sampling_summary.get("split_manifest_sha256"))
if TRAINING_SAMPLING_MODE == "undersampled" and not sampling_summary.get("split_manifest_sha256"):
    raise RuntimeError("No fixed split manifest found. Run prepare_undersampled_split.ipynb first.")
train_ds, validation_ds, test_ds = make_datasets(
    batch_size=4,
    seed=SEED,
    training_sampling_mode=TRAINING_SAMPLING_MODE,
)
images, labels = next(iter(train_ds))

print("Image batch shape:", images.shape)
print("Image dtype:", images.dtype)
print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))
print("Labels in first batch:", labels.numpy())

ImportError: cannot import name 'TRAIN_RATIO' from 'configs.config' (D:\M L\Diabetic_Retinopathy_Detection_mini_project\configs\config.py)

## 3. EfficientNet — model initialization preview

In [ ]:
from scripts.train_efficientnet import build_efficientnet_model

efficientnet_preview = build_efficientnet_model(weights=None)
efficientnet_preview.summary()
print("Input:", efficientnet_preview.input_shape)
print("Output:", efficientnet_preview.output_shape)
del efficientnet_preview
tf.keras.backend.clear_session()

## 4. EfficientNet — manually controlled training
Choose `scratch` for a new isolated run or `resume` to continue the newest valid checkpoint. The epoch input is the **total target epoch**, not the number of additional epochs.

In [ ]:
from scripts.train_efficientnet import train_efficientnet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or rerun and explicitly opt into CPU training.")
else:
    run_mode = input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = input("Batch size [4]: ").strip()
    epochs_text = input("Total target epochs [10]: ").strip()

    batch_size = int(batch_text or 4)
    target_epochs = int(epochs_text or 10)

    learning_rate = float(input("Enter value for learning rate: (Default : 0.001) "))

    efficientnet_result = train_efficientnet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )

In [ ]:
from scripts.utils import print_run_status

selected_model_key = training_model_name("efficientnet", TRAINING_SAMPLING_MODE)
print_run_status(selected_model_key)


## EfficientNetB0 — Baseline

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy |
| Training Metric | Sparse Categorical Accuracy |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | None (Unweighted Baseline) |
| Batch Size | 4 (default; configurable per run) |
| Target Epochs | 10 (default; configurable per run) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest Validation Loss |


**Training data strategy:** the standalone `prepare_undersampled_split.ipynb` has already fixed the training subset at a maximum of 2,500 examples per class. Training scripts do not choose samples. Class weights (when enabled) are calculated from the manifest class counts. Validation and test are also fixed sampled subsets from the same 70:10:20 manifest.


## 6. Class-weighted EfficientNet — manually controlled training

**Run only after the unweighted EfficientNet baseline experiment you intend to compare against.** This is a separate experiment with its own run folders and checkpoints; it does not initialize from the trained unweighted model. It uses the same EfficientNetB0/ImageNet architecture and the same fixed `btgraham-300` split manifest.

Balanced class weights are calculated from the selected training subset only using `w_c = N / (K × n_c)`. They are applied to training examples only. Validation loss remains unweighted for model selection, and the test split is not evaluated here. Choose `scratch` for an independent new run or `resume` to continue the latest weighted-run checkpoint. The Epoch value is the **total target epoch**, not the number of additional epochs.


In [ ]:
from scripts.train_efficientnet_weighted import train_efficientnet_weighted

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False

if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or rerun and explicitly opt into CPU training.")
else:
    run_mode = input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = input("Batch size [4]: ").strip()
    epoch_text = input("Total target Epoch [10]: ").strip()

    batch_size = int(batch_text or 4)
    target_epochs = int(epoch_text or 10)

    learning_rate = float(input("Enter value for learning rate: (Default : 0.001) "))

    weighted_efficientnet_result = train_efficientnet_weighted(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )

In [ ]:
from scripts.utils import print_run_status

selected_model_key = training_model_name("efficientnet_weighted", TRAINING_SAMPLING_MODE)
print_run_status(selected_model_key)

### Class-weighted EfficientNet — current hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy with per-example sample weights |
| Training Metric | Sparse Categorical Accuracy (unweighted reporting metric) |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | Balanced inverse-frequency: \(w_c = N/(K n_c)\), calculated from the selected training subset only |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |


**Training data strategy:** the standalone `prepare_undersampled_split.ipynb` has already fixed the training subset at a maximum of 2,500 examples per class. Training scripts do not choose samples. Class weights (when enabled) are calculated from the manifest class counts. Validation and test are also fixed sampled subsets from the same 70:10:20 manifest.


## 7. Weighted DenseNet121 — preview and training

This experiment uses a frozen ImageNet-pretrained DenseNet121 backbone, global average pooling, dropout, and a five-class softmax head. Balanced inverse-frequency class weights are computed from the selected **training subset only** and applied to training loss only. Validation remains unweighted and uses the saved 70:10:20 subset. It has independent checkpoints and run folders under the mode-specific model root (for example, `models/densenet_undersampled/`); it does not load EfficientNet weights. The total target Epoch defaults to 10 and batch size defaults to 4.

In [ ]:
from scripts.train_densenet import build_densenet_model

densenet_preview = build_densenet_model(weights=None)
densenet_preview.summary()
print("Input:", densenet_preview.input_shape)
print("Output:", densenet_preview.output_shape)
del densenet_preview
tf.keras.backend.clear_session()

### Weighted DenseNet — manually controlled training

Only run this cell when ready to train. If the model runs out of GPU memory, reduce batch size before changing the image size.

In [ ]:
from scripts.train_densenet import train_densenet

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False
if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or explicitly opt into CPU training.")
else:
    run_mode = input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = input("Batch size [4]: ").strip()
    epoch_text = input("Total target Epoch [10]: ").strip()
    batch_size = int(batch_text or 4)
    target_epochs = int(epoch_text or 10)
    learning_rate = float(input("Enter value for learning rate: (Default : 0.001) "))
    densenet_result = train_densenet(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )

### Weighted DenseNet — inspect run status

Lists only this model's runs; it does not load a model or evaluate it.

In [ ]:
from scripts.utils import print_run_status

selected_model_key = training_model_name("densenet", TRAINING_SAMPLING_MODE)
print_run_status(selected_model_key)

### Weighted DenseNet121 — hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | DenseNet121 |
| Pretrained Weights | ImageNet |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbone Trainable | False (Frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy with training sample weights |
| Training Metric | Sparse Categorical Accuracy (unweighted reporting metric) |
| Dropout Rate | 0.30 (30%) |
| Classification Head | DenseNet preprocessing → Global Average Pooling → Dropout → Dense (5, Softmax) |
| Class Weighting | Balanced inverse-frequency: `N / (K × n_c)`, selected training subset only |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |

## 8. EfficientNetB0 + DenseNet121 feature-fusion model

The starter fusion experiment concatenates the two branches' global-average-pooled features, followed by dropout and a five-class softmax head. Both ImageNet feature extractors are frozen for this first resource-conscious version. This is **feature concatenation**, not attention fusion and not averaging final predictions. This initial fusion run is unweighted. Because it contains two backbones, batch size 4 may exceed 4 GB VRAM; reduce the batch size first if needed.

In [ ]:
from scripts.train_fusion import build_fusion_model

fusion_preview = build_fusion_model(weights=None)
fusion_preview.summary()
print("Input:", fusion_preview.input_shape)
print("Output:", fusion_preview.output_shape)
del fusion_preview
tf.keras.backend.clear_session()

### Fusion — manually controlled training

Training is separate from the EfficientNet and DenseNet run roots. Use `scratch` for a new independent fusion run or `resume` to restore its own latest valid checkpoint.

In [ ]:
from scripts.train_fusion import train_fusion

available_gpus = tf.config.list_physical_devices("GPU")
allow_cpu = False
if not available_gpus:
    print("WARNING: TensorFlow cannot currently see a GPU.")
    cpu_choice = input("To deliberately allow potentially slow CPU training, type RUN-ON-CPU: ").strip()
    allow_cpu = cpu_choice == "RUN-ON-CPU"

if not available_gpus and not allow_cpu:
    print("Stopped without training. Fix GPU visibility, or explicitly opt into CPU training.")
else:
    run_mode = input("Run mode [scratch/resume]: ").strip().lower()
    batch_text = input("Batch size [4]: ").strip()
    epoch_text = input("Total target Epoch [10]: ").strip()
    batch_size = int(batch_text or 4)
    target_epochs = int(epoch_text or 10)
    learning_rate = float(input("Enter value for learning rate: (Default : 0.001) "))
    fusion_result = train_fusion(
        run_mode=run_mode,
        batch_size=batch_size,
        epochs=target_epochs,
        allow_cpu=allow_cpu,
        learning_rate=learning_rate,
        training_sampling_mode=TRAINING_SAMPLING_MODE
    )

### Fusion — inspect run status

In [ ]:
from scripts.utils import print_run_status

selected_model_key = training_model_name("fusion", TRAINING_SAMPLING_MODE)
print_run_status(selected_model_key)

### Fusion — current hyperparameters

| Parameter | Current Setting |
|---|---|
| Model Architecture | EfficientNetB0 + DenseNet121 feature concatenation |
| Pretrained Weights | ImageNet for both backbones |
| Input Image Size | 300 × 300 × 3 |
| Number of Classes | 5 |
| Backbones Trainable | False (both frozen) |
| Optimizer | Adam |
| Learning Rate | 0.001 (1e-3) |
| Loss Function | Sparse Categorical Cross-Entropy |
| Training Metric | Sparse Categorical Accuracy |
| Dropout Rate | 0.30 (30%) |
| Classification Head | Concatenate pooled features → Dropout → Dense (5, Softmax) |
| Class Weighting | None (initial fusion experiment) |
| Batch Size | 4 (default; configurable per run) |
| Epoch | User-defined total target (prompt default: 10) |
| Random Seed | 42 |
| Shuffle Buffer | 2048 |
| Best Model Selection | Lowest unweighted validation loss |
